# Standalone Offline Controller Training

This notebook is self-contained for Google Colab. It trains **BC**, **AWAC**, **IQL**, and **TD3+BC** from a dataset folder that you provide. It does not import from this repository's Controllers folder.

Expected dataset files are .h5, .hdf5, or .npz files with common transition keys such as observations, actions, rewards, terminals, and optionally next_observations.


## 1. Colab Setup

Run this first. Colab already includes PyTorch, NumPy, and h5py in most runtimes. If you want GPU training, use Runtime > Change runtime type > GPU before training.


In [ ]:
from __future__ import annotations

import copy
import json
import math
import os
import random
import shutil
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import CosineAnnealingLR

try:
    import h5py
except ImportError as exc:
    raise ImportError("h5py is required. In a notebook cell, run: %pip install h5py") from exc

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 2. Provide Your Dataset

Set DATA_FOLDER to the folder that contains your .h5, .hdf5, or .npz transition files.

Common Colab options:

- Upload a zip, then call upload_and_extract_zip().
- Mount Google Drive with mount_google_drive(), then set DATA_FOLDER to a Drive path.
- Manually upload files into /content/dataset using the Colab file browser.


In [ ]:
DATA_FOLDER = Path("/content/dataset")
CHECKPOINT_ROOT = Path("/content/offline_controller_runs")

def mount_google_drive() -> None:
    from google.colab import drive
    drive.mount("/content/drive")

def upload_and_extract_zip(extract_to: str | Path = "/content/dataset") -> Path:
    from google.colab import files
    uploaded = files.upload()
    zip_paths = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
    if not zip_paths:
        raise ValueError("Upload a .zip file that contains your dataset files.")

    extract_to = Path(extract_to)
    extract_to.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_paths[0], "r") as archive:
        archive.extractall(extract_to)
    print(f"Extracted {zip_paths[0]} to {extract_to}")
    return extract_to

print(f"DATA_FOLDER is currently: {DATA_FOLDER}")
print("Change DATA_FOLDER above if your dataset is somewhere else.")


## 3. Data Loader

This loader searches the dataset folder for .h5, .hdf5, and .npz files. It accepts several common key names for observations, actions, rewards, terminals, and next observations. If next_observations is missing, it is computed by shifting observations by one step and holding terminal states in place.


In [ ]:
OBS_KEYS = ["observations", "obs", "states", "state", "o", "x", "lidar"]
NEXT_OBS_KEYS = ["next_observations", "next_obs", "next_states"]
ACTION_KEYS = ["actions", "action", "act", "a", "u", "controls", "t_action", "steering_angle"]
REWARD_KEYS = ["rewards", "reward", "r"]
DONE_KEYS = ["terminals", "dones", "done", "timeouts"]

def find_h5_key(h5: h5py.File, candidates: Sequence[str]) -> Optional[str]:
    for key in candidates:
        if key in h5:
            return key

    def walk(group: h5py.Group, prefix: str = "") -> Optional[str]:
        for name, item in group.items():
            path = f"{prefix}/{name}" if prefix else name
            if isinstance(item, h5py.Dataset) and name in candidates:
                return path
            if isinstance(item, h5py.Group):
                found = walk(item, path)
                if found is not None:
                    return found
        return None

    return walk(h5)

def read_h5_array(h5: h5py.File, path: str) -> np.ndarray:
    item = h5
    for part in path.split("/"):
        item = item[part]
    return np.asarray(item)

def find_npz_key(npz: np.lib.npyio.NpzFile, candidates: Sequence[str]) -> Optional[str]:
    for key in candidates:
        if key in npz.files:
            return key
    return None

def as_2d(array: np.ndarray) -> np.ndarray:
    array = np.asarray(array)
    if array.ndim == 1:
        return array[:, None]
    return array

def compute_next_observations(observations: np.ndarray, terminals: np.ndarray) -> np.ndarray:
    next_observations = observations.copy()
    if len(observations) > 1:
        next_observations[:-1] = observations[1:]
        terminal_mask = np.asarray(terminals).reshape(-1).astype(bool)
        next_observations[terminal_mask] = observations[terminal_mask]
    return next_observations

def dataset_paths(data_folder: str | Path) -> List[Path]:
    folder = Path(data_folder).expanduser()
    if not folder.exists():
        raise FileNotFoundError(f"Dataset folder does not exist: {folder}")

    paths = sorted(folder.glob("*.h5")) + sorted(folder.glob("*.hdf5"))
    h5_stems = {(path.parent, path.stem) for path in paths}
    paths.extend(path for path in sorted(folder.glob("*.npz")) if (path.parent, path.stem) not in h5_stems)

    if not paths:
        paths = sorted(folder.rglob("*.h5")) + sorted(folder.rglob("*.hdf5"))
        h5_stems = {(path.parent, path.stem) for path in paths}
        paths.extend(path for path in sorted(folder.rglob("*.npz")) if (path.parent, path.stem) not in h5_stems)

    if not paths:
        raise FileNotFoundError(f"No .h5, .hdf5, or .npz files found in {folder}")
    return paths

def load_npz_transitions(
    path: Path,
    obs_key: Optional[str] = None,
    action_key: Optional[str] = None,
    reward_key: Optional[str] = None,
    done_key: Optional[str] = None,
    next_obs_key: Optional[str] = None,
) -> Dict[str, np.ndarray]:
    with np.load(path, allow_pickle=False) as data:
        ok = obs_key or find_npz_key(data, OBS_KEYS)
        ak = action_key or find_npz_key(data, ACTION_KEYS)
        rk = reward_key or find_npz_key(data, REWARD_KEYS)
        dk = done_key or find_npz_key(data, DONE_KEYS)
        nok = next_obs_key or find_npz_key(data, NEXT_OBS_KEYS)
        if ok is None or ak is None:
            raise KeyError(f"{path} is missing observation or action keys. Keys: {data.files}")

        observations = as_2d(data[ok]).astype(np.float32)
        actions = as_2d(data[ak]).astype(np.float32)
        rewards = np.asarray(data[rk]).reshape(-1).astype(np.float32) if rk is not None else np.zeros(len(observations), dtype=np.float32)
        terminals = np.asarray(data[dk]).reshape(-1).astype(np.float32) if dk is not None else np.zeros(len(observations), dtype=np.float32)
        next_observations = as_2d(data[nok]).astype(np.float32) if nok is not None else compute_next_observations(observations, terminals)

    n = min(len(observations), len(actions), len(rewards), len(terminals), len(next_observations))
    return {
        "observations": observations[:n],
        "actions": actions[:n],
        "rewards": rewards[:n],
        "terminals": terminals[:n],
        "next_observations": next_observations[:n],
    }

def load_h5_transitions(
    data_folder: str | Path,
    obs_key: Optional[str] = None,
    action_key: Optional[str] = None,
    reward_key: Optional[str] = None,
    done_key: Optional[str] = None,
    next_obs_key: Optional[str] = None,
) -> Dict[str, np.ndarray]:
    obs_list, action_list, reward_list, done_list, next_obs_list = [], [], [], [], []
    paths = dataset_paths(data_folder)
    for path in paths:
        if path.suffix.lower() == ".npz":
            loaded = load_npz_transitions(path, obs_key, action_key, reward_key, done_key, next_obs_key)
            observations = loaded["observations"]
            actions = loaded["actions"]
            rewards = loaded["rewards"]
            terminals = loaded["terminals"]
            next_observations = loaded["next_observations"]
        else:
            with h5py.File(path, "r") as h5:
                ok = obs_key or find_h5_key(h5, OBS_KEYS)
                ak = action_key or find_h5_key(h5, ACTION_KEYS)
                rk = reward_key or find_h5_key(h5, REWARD_KEYS)
                dk = done_key or find_h5_key(h5, DONE_KEYS)
                nok = next_obs_key or find_h5_key(h5, NEXT_OBS_KEYS)
                if ok is None or ak is None:
                    raise KeyError(f"{path} is missing observation or action keys. Top-level keys: {list(h5.keys())}")

                observations = as_2d(read_h5_array(h5, ok)).astype(np.float32)
                actions = as_2d(read_h5_array(h5, ak)).astype(np.float32)
                rewards = np.asarray(read_h5_array(h5, rk)).reshape(-1).astype(np.float32) if rk is not None else np.zeros(len(observations), dtype=np.float32)
                terminals = np.asarray(read_h5_array(h5, dk)).reshape(-1).astype(np.float32) if dk is not None else np.zeros(len(observations), dtype=np.float32)
                next_observations = as_2d(read_h5_array(h5, nok)).astype(np.float32) if nok is not None else compute_next_observations(observations, terminals)

        n = min(len(observations), len(actions), len(rewards), len(terminals), len(next_observations))
        obs_list.append(observations[:n])
        action_list.append(actions[:n])
        reward_list.append(rewards[:n])
        done_list.append(terminals[:n])
        next_obs_list.append(next_observations[:n])

    dataset = {
        "observations": np.concatenate(obs_list, axis=0),
        "actions": np.concatenate(action_list, axis=0),
        "rewards": np.concatenate(reward_list, axis=0),
        "terminals": np.concatenate(done_list, axis=0),
        "next_observations": np.concatenate(next_obs_list, axis=0),
    }
    print(f"Loaded {len(dataset['observations'])} transitions from {len(paths)} files.")
    return dataset

def preview_dataset(data_folder: str | Path) -> Dict[str, float | int | str]:
    dataset = load_h5_transitions(data_folder)
    summary = {
        "data_folder": str(Path(data_folder)),
        "num_transitions": int(dataset["observations"].shape[0]),
        "obs_dim": int(dataset["observations"].shape[1]),
        "action_dim": int(dataset["actions"].shape[1]),
        "reward_min": float(np.min(dataset["rewards"])),
        "reward_mean": float(np.mean(dataset["rewards"])),
        "reward_max": float(np.max(dataset["rewards"])),
    }
    print(json.dumps(summary, indent=2))
    return summary


## 4. Models and Replay Buffer

These are the shared neural network and replay buffer pieces used by all four algorithms.


In [ ]:
def set_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def normalize_dataset(dataset: Dict[str, np.ndarray], normalize_obs: bool, normalize_actions: bool) -> Dict[str, np.ndarray]:
    stats = {
        "obs_mean": None,
        "obs_std": None,
        "action_mean": None,
        "action_std": None,
    }
    if normalize_obs:
        stats["obs_mean"] = dataset["observations"].mean(axis=0, keepdims=True)
        stats["obs_std"] = dataset["observations"].std(axis=0, keepdims=True) + 1e-6
        dataset["observations"] = (dataset["observations"] - stats["obs_mean"]) / stats["obs_std"]
        dataset["next_observations"] = (dataset["next_observations"] - stats["obs_mean"]) / stats["obs_std"]
    if normalize_actions:
        stats["action_mean"] = dataset["actions"].mean(axis=0, keepdims=True)
        stats["action_std"] = dataset["actions"].std(axis=0, keepdims=True) + 1e-6
        dataset["actions"] = (dataset["actions"] - stats["action_mean"]) / stats["action_std"]
    return stats

class ReplayBuffer:
    def __init__(self, dataset: Dict[str, np.ndarray], device: str = "cpu"):
        self.device = torch.device(device)
        self.observations = torch.as_tensor(dataset["observations"], dtype=torch.float32, device=self.device)
        self.actions = torch.as_tensor(dataset["actions"], dtype=torch.float32, device=self.device)
        self.rewards = torch.as_tensor(dataset["rewards"], dtype=torch.float32, device=self.device).view(-1, 1)
        self.terminals = torch.as_tensor(dataset["terminals"], dtype=torch.float32, device=self.device).view(-1, 1)
        self.next_observations = torch.as_tensor(dataset["next_observations"], dtype=torch.float32, device=self.device)
        self.size = self.observations.shape[0]
        self.obs_dim = self.observations.shape[1]
        self.action_dim = self.actions.shape[1]

    def sample(self, batch_size: int) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        indices = torch.randint(0, self.size, (batch_size,), device=self.device)
        return (
            self.observations[indices],
            self.actions[indices],
            self.rewards[indices],
            self.next_observations[indices],
            self.terminals[indices],
        )

class MLP(nn.Module):
    def __init__(self, dims: Sequence[int], activation=nn.ReLU, output_activation=None):
        super().__init__()
        layers: List[nn.Module] = []
        for i in range(len(dims) - 2):
            layers.extend([nn.Linear(dims[i], dims[i + 1]), activation()])
        layers.append(nn.Linear(dims[-2], dims[-1]))
        if output_activation is not None:
            layers.append(output_activation())
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class DeterministicActor(nn.Module):
    def __init__(self, obs_dim: int, action_dim: int, hidden_dim: int = 256, hidden_layers: int = 2):
        super().__init__()
        self.net = MLP([obs_dim, *([hidden_dim] * hidden_layers), action_dim], output_activation=nn.Tanh)

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        return self.net(obs)

class GaussianActor(nn.Module):
    def __init__(self, obs_dim: int, action_dim: int, hidden_dim: int = 256, hidden_layers: int = 2):
        super().__init__()
        self.mean = MLP([obs_dim, *([hidden_dim] * hidden_layers), action_dim], output_activation=nn.Tanh)
        self.log_std = nn.Parameter(torch.zeros(action_dim, dtype=torch.float32))

    def forward(self, obs: torch.Tensor) -> torch.distributions.Normal:
        std = torch.exp(self.log_std.clamp(-20.0, 2.0))
        return torch.distributions.Normal(self.mean(obs), std)

    def sample(self, obs: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        dist = self(obs)
        action = dist.rsample()
        return action.clamp(-1.0, 1.0), dist.log_prob(action).sum(dim=-1, keepdim=True)

class TwinQ(nn.Module):
    def __init__(self, obs_dim: int, action_dim: int, hidden_dim: int = 256, hidden_layers: int = 2):
        super().__init__()
        dims = [obs_dim + action_dim, *([hidden_dim] * hidden_layers), 1]
        self.q1 = MLP(dims)
        self.q2 = MLP(dims)

    def both(self, obs: torch.Tensor, action: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        sa = torch.cat([obs, action], dim=-1)
        return self.q1(sa), self.q2(sa)

    def forward(self, obs: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
        q1, q2 = self.both(obs, action)
        return torch.min(q1, q2)

class ValueFunction(nn.Module):
    def __init__(self, obs_dim: int, hidden_dim: int = 256, hidden_layers: int = 2):
        super().__init__()
        self.net = MLP([obs_dim, *([hidden_dim] * hidden_layers), 1])

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        return self.net(obs)

def soft_update(target: nn.Module, source: nn.Module, tau: float) -> None:
    for target_param, source_param in zip(target.parameters(), source.parameters()):
        target_param.data.copy_((1.0 - tau) * target_param.data + tau * source_param.data)

def json_ready(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")

def prepare_checkpoint_dir(path: str | Path, config) -> Path:
    checkpoint_dir = Path(path)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    with open(checkpoint_dir / "config.json", "w", encoding="utf-8") as handle:
        json.dump(asdict(config), handle, indent=2, default=json_ready)
    return checkpoint_dir

def save_checkpoint(path: Path, payload: Dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, path)


## 5. Training Algorithms

Each trainer loads the dataset from config.data_folder, normalizes it if requested, and saves checkpoints into config.checkpoint_dir.


In [ ]:
EXP_ADV_MAX = 100.0

@dataclass
class TrainConfig:
    data_folder: str
    checkpoint_dir: str
    device: str = DEVICE
    seed: int = 0
    max_steps: int = 50_000
    batch_size: int = 256
    eval_freq: int = 5_000
    discount: float = 0.99
    tau: float = 0.005
    normalize_obs: bool = True
    normalize_actions: bool = False
    hidden_dim: int = 256
    hidden_layers: int = 2
    lr: float = 3e-4
    actor_lr: float = 3e-4
    critic_lr: float = 3e-4
    vf_lr: float = 3e-4
    qf_lr: float = 3e-4
    awac_lambda: float = 1.0
    max_weight: float = 20.0
    beta: float = 3.0
    iql_tau: float = 0.7
    deterministic_actor: bool = False
    policy_noise: float = 0.2
    noise_clip: float = 0.5
    policy_delay: int = 2
    alpha: float = 2.5

def make_buffer(config: TrainConfig) -> Tuple[ReplayBuffer, Dict[str, np.ndarray]]:
    dataset = load_h5_transitions(config.data_folder)
    stats = normalize_dataset(dataset, config.normalize_obs, config.normalize_actions)
    return ReplayBuffer(dataset, config.device), stats

def train_bc(config: TrainConfig) -> Path:
    set_seed(config.seed)
    checkpoint_dir = prepare_checkpoint_dir(config.checkpoint_dir, config)
    buffer, stats = make_buffer(config)

    actor = DeterministicActor(buffer.obs_dim, buffer.action_dim, config.hidden_dim, config.hidden_layers).to(config.device)
    optimizer = torch.optim.Adam(actor.parameters(), lr=config.lr)
    best_loss = float("inf")

    for step in range(1, config.max_steps + 1):
        observations, actions, *_ = buffer.sample(config.batch_size)
        predictions = actor(observations)
        loss = F.mse_loss(predictions, actions)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % config.eval_freq == 0 or step == 1:
            print(f"step={step} bc_loss={loss.item():.6f}")
            payload = {"actor": actor.state_dict(), "optimizer": optimizer.state_dict(), "step": step, "loss": loss.item(), "stats": stats}
            save_checkpoint(checkpoint_dir / f"checkpoint_{step}.pt", payload)
            if loss.item() < best_loss:
                best_loss = loss.item()
                save_checkpoint(checkpoint_dir / "best_bc.pt", payload)

    final_path = checkpoint_dir / "final_bc.pt"
    save_checkpoint(final_path, {"actor": actor.state_dict(), "step": config.max_steps, "stats": stats})
    return final_path

def train_awac(config: TrainConfig) -> Path:
    set_seed(config.seed)
    checkpoint_dir = prepare_checkpoint_dir(config.checkpoint_dir, config)
    buffer, stats = make_buffer(config)

    actor = GaussianActor(buffer.obs_dim, buffer.action_dim, config.hidden_dim, config.hidden_layers).to(config.device)
    critic = TwinQ(buffer.obs_dim, buffer.action_dim, config.hidden_dim, config.hidden_layers).to(config.device)
    critic_target = copy.deepcopy(critic).requires_grad_(False).to(config.device)
    actor_optimizer = torch.optim.Adam(actor.parameters(), lr=config.actor_lr)
    critic_optimizer = torch.optim.Adam(critic.parameters(), lr=config.critic_lr)
    best_actor_loss = float("inf")

    for step in range(1, config.max_steps + 1):
        observations, actions, rewards, next_observations, terminals = buffer.sample(config.batch_size)

        with torch.no_grad():
            next_actions, _ = actor.sample(next_observations)
            target_q = critic_target(next_observations, next_actions)
            target = rewards + (1.0 - terminals) * config.discount * target_q

        q1, q2 = critic.both(observations, actions)
        critic_loss = F.mse_loss(q1, target) + F.mse_loss(q2, target)
        critic_optimizer.zero_grad()
        critic_loss.backward()
        critic_optimizer.step()
        soft_update(critic_target, critic, config.tau)

        with torch.no_grad():
            sampled_actions, _ = actor.sample(observations)
            v = critic(observations, sampled_actions)
            q = critic(observations, actions)
            weights = torch.exp((q - v) / config.awac_lambda).clamp(max=config.max_weight)

        dist = actor(observations)
        log_prob = dist.log_prob(actions).sum(dim=-1, keepdim=True)
        actor_loss = -(weights * log_prob).mean()
        actor_optimizer.zero_grad()
        actor_loss.backward()
        actor_optimizer.step()

        if step % config.eval_freq == 0 or step == 1:
            print(f"step={step} critic_loss={critic_loss.item():.6f} actor_loss={actor_loss.item():.6f}")
            payload = {
                "actor": actor.state_dict(),
                "critic": critic.state_dict(),
                "actor_optimizer": actor_optimizer.state_dict(),
                "critic_optimizer": critic_optimizer.state_dict(),
                "step": step,
                "stats": stats,
            }
            save_checkpoint(checkpoint_dir / f"checkpoint_{step}.pt", payload)
            if actor_loss.item() < best_actor_loss:
                best_actor_loss = actor_loss.item()
                save_checkpoint(checkpoint_dir / "best_awac.pt", payload)

    final_path = checkpoint_dir / "final_awac.pt"
    save_checkpoint(final_path, {"actor": actor.state_dict(), "critic": critic.state_dict(), "step": config.max_steps, "stats": stats})
    return final_path

def asymmetric_l2_loss(error: torch.Tensor, tau: float) -> torch.Tensor:
    weight = torch.abs(tau - (error < 0).float())
    return (weight * error.pow(2)).mean()

def actor_bc_loss(actor, observations: torch.Tensor, actions: torch.Tensor) -> torch.Tensor:
    output = actor(observations)
    if isinstance(output, torch.distributions.Distribution):
        return -output.log_prob(actions).sum(dim=-1, keepdim=True)
    return (output - actions).pow(2).sum(dim=-1, keepdim=True)

def train_iql(config: TrainConfig) -> Path:
    set_seed(config.seed)
    checkpoint_dir = prepare_checkpoint_dir(config.checkpoint_dir, config)
    buffer, stats = make_buffer(config)

    qf = TwinQ(buffer.obs_dim, buffer.action_dim, config.hidden_dim, config.hidden_layers).to(config.device)
    q_target = copy.deepcopy(qf).requires_grad_(False).to(config.device)
    vf = ValueFunction(buffer.obs_dim, config.hidden_dim, config.hidden_layers).to(config.device)
    actor_cls = DeterministicActor if config.deterministic_actor else GaussianActor
    actor = actor_cls(buffer.obs_dim, buffer.action_dim, config.hidden_dim, config.hidden_layers).to(config.device)

    q_optimizer = torch.optim.Adam(qf.parameters(), lr=config.qf_lr)
    v_optimizer = torch.optim.Adam(vf.parameters(), lr=config.vf_lr)
    actor_optimizer = torch.optim.Adam(actor.parameters(), lr=config.actor_lr)
    actor_scheduler = CosineAnnealingLR(actor_optimizer, config.max_steps)

    for step in range(1, config.max_steps + 1):
        observations, actions, rewards, next_observations, terminals = buffer.sample(config.batch_size)

        with torch.no_grad():
            target_q = q_target(observations, actions)
        v = vf(observations)
        advantage = target_q - v
        v_loss = asymmetric_l2_loss(advantage, config.iql_tau)
        v_optimizer.zero_grad()
        v_loss.backward()
        v_optimizer.step()

        with torch.no_grad():
            next_v = vf(next_observations)
            q_target_value = rewards + (1.0 - terminals) * config.discount * next_v
        q1, q2 = qf.both(observations, actions)
        q_loss = F.mse_loss(q1, q_target_value) + F.mse_loss(q2, q_target_value)
        q_optimizer.zero_grad()
        q_loss.backward()
        q_optimizer.step()
        soft_update(q_target, qf, config.tau)

        weights = torch.exp(config.beta * advantage.detach()).clamp(max=EXP_ADV_MAX)
        policy_loss = (weights * actor_bc_loss(actor, observations, actions)).mean()
        actor_optimizer.zero_grad()
        policy_loss.backward()
        actor_optimizer.step()
        actor_scheduler.step()

        if step % config.eval_freq == 0 or step == 1:
            print(f"step={step} value_loss={v_loss.item():.6f} q_loss={q_loss.item():.6f} actor_loss={policy_loss.item():.6f}")
            save_checkpoint(
                checkpoint_dir / f"checkpoint_{step}.pt",
                {
                    "actor": actor.state_dict(),
                    "qf": qf.state_dict(),
                    "vf": vf.state_dict(),
                    "q_optimizer": q_optimizer.state_dict(),
                    "v_optimizer": v_optimizer.state_dict(),
                    "actor_optimizer": actor_optimizer.state_dict(),
                    "actor_scheduler": actor_scheduler.state_dict(),
                    "step": step,
                    "stats": stats,
                },
            )

    final_path = checkpoint_dir / "final_iql.pt"
    save_checkpoint(final_path, {"actor": actor.state_dict(), "qf": qf.state_dict(), "vf": vf.state_dict(), "step": config.max_steps, "stats": stats})
    return final_path

def train_td3bc(config: TrainConfig) -> Path:
    set_seed(config.seed)
    checkpoint_dir = prepare_checkpoint_dir(config.checkpoint_dir, config)
    buffer, stats = make_buffer(config)

    actor = DeterministicActor(buffer.obs_dim, buffer.action_dim, config.hidden_dim, config.hidden_layers).to(config.device)
    actor_target = copy.deepcopy(actor).requires_grad_(False).to(config.device)
    critic = TwinQ(buffer.obs_dim, buffer.action_dim, config.hidden_dim, config.hidden_layers).to(config.device)
    critic_target = copy.deepcopy(critic).requires_grad_(False).to(config.device)
    actor_optimizer = torch.optim.Adam(actor.parameters(), lr=config.actor_lr)
    critic_optimizer = torch.optim.Adam(critic.parameters(), lr=config.critic_lr)
    best_action_mse = float("inf")

    for step in range(1, config.max_steps + 1):
        observations, actions, rewards, next_observations, terminals = buffer.sample(config.batch_size)

        with torch.no_grad():
            noise = (torch.randn_like(actions) * config.policy_noise).clamp(-config.noise_clip, config.noise_clip)
            next_actions = (actor_target(next_observations) + noise).clamp(-1.0, 1.0)
            target_q = critic_target(next_observations, next_actions)
            target = rewards + (1.0 - terminals) * config.discount * target_q

        q1, q2 = critic.both(observations, actions)
        critic_loss = F.mse_loss(q1, target) + F.mse_loss(q2, target)
        critic_optimizer.zero_grad()
        critic_loss.backward()
        critic_optimizer.step()

        actor_loss_value = torch.tensor(0.0, device=config.device)
        if step % config.policy_delay == 0:
            policy_actions = actor(observations)
            q_policy = critic.q1(torch.cat([observations, policy_actions], dim=-1))
            lam = config.alpha / (q_policy.abs().mean().detach() + 1e-6)
            bc_loss = F.mse_loss(policy_actions, actions)
            actor_loss = -lam * q_policy.mean() + bc_loss
            actor_loss_value = actor_loss.detach()

            actor_optimizer.zero_grad()
            actor_loss.backward()
            actor_optimizer.step()
            soft_update(actor_target, actor, config.tau)
            soft_update(critic_target, critic, config.tau)

        if step % config.eval_freq == 0 or step == 1:
            with torch.no_grad():
                eval_observations, eval_actions, *_ = buffer.sample(config.batch_size)
                action_mse = F.mse_loss(actor(eval_observations), eval_actions).item()
            print(f"step={step} critic_loss={critic_loss.item():.6f} actor_loss={actor_loss_value.item():.6f} action_mse={action_mse:.6f}")
            payload = {
                "actor": actor.state_dict(),
                "critic": critic.state_dict(),
                "actor_optimizer": actor_optimizer.state_dict(),
                "critic_optimizer": critic_optimizer.state_dict(),
                "step": step,
                "stats": stats,
            }
            save_checkpoint(checkpoint_dir / f"checkpoint_{step}.pt", payload)
            if action_mse < best_action_mse:
                best_action_mse = action_mse
                save_checkpoint(checkpoint_dir / "best_td3_bc.pt", payload)

    final_path = checkpoint_dir / "final_td3_bc.pt"
    save_checkpoint(final_path, {"actor": actor.state_dict(), "critic": critic.state_dict(), "step": config.max_steps, "stats": stats})
    return final_path


## 6. Model Notes

**BC** imitates dataset actions directly with MSE and is the fastest baseline.

**AWAC** uses critic-estimated advantages to weight behavior-cloning updates. Tune awac_lambda first.

**IQL** uses expectile value learning and advantage-weighted actor updates. Tune iql_tau and beta first.

**TD3+BC** combines TD3-style critic learning with a behavior-cloning actor penalty. Tune alpha first.


## 7. Configure Run

Edit this cell before training. Keep max_steps small for a smoke test, then increase it for a full run. You can also pass epochs to train_one() below to compute steps from the dataset size.


In [ ]:
PREVIEW_DATASET = True

DEFAULTS = {
    "bc": dict(batch_size=1024, max_steps=50_000, eval_freq=5_000, lr=3e-4),
    "awac": dict(batch_size=256, max_steps=100_000, eval_freq=5_000, actor_lr=3e-4, critic_lr=3e-4, awac_lambda=1.0, max_weight=20.0),
    "iql": dict(batch_size=256, max_steps=100_000, eval_freq=5_000, actor_lr=3e-4, qf_lr=3e-4, vf_lr=3e-4, iql_tau=0.7, beta=3.0),
    "td3bc": dict(batch_size=512, max_steps=50_000, eval_freq=5_000, actor_lr=3e-4, critic_lr=1e-4, alpha=2.5),
}

TRAINERS = {
    "bc": train_bc,
    "awac": train_awac,
    "iql": train_iql,
    "td3bc": train_td3bc,
}

FINAL_NAMES = {
    "bc": "final_bc.pt",
    "awac": "final_awac.pt",
    "iql": "final_iql.pt",
    "td3bc": "final_td3_bc.pt",
}

def steps_for_epochs(num_transitions: int, batch_size: int, epochs: int) -> int:
    return int(math.ceil(num_transitions / float(batch_size)) * epochs)

def make_config(algorithm: str, epochs: Optional[int] = None, **overrides) -> TrainConfig:
    algorithm = algorithm.lower()
    if algorithm not in TRAINERS:
        raise ValueError(f"Unknown algorithm {algorithm!r}. Use one of {list(TRAINERS)}")

    values = dict(
        data_folder=str(DATA_FOLDER),
        checkpoint_dir=str(CHECKPOINT_ROOT / algorithm),
        device=DEVICE,
        seed=0,
        normalize_obs=True,
        normalize_actions=False,
        hidden_dim=256,
        hidden_layers=2,
        discount=0.99,
        tau=0.005,
    )
    values.update(DEFAULTS[algorithm])
    values.update(overrides)

    if epochs is not None:
        dataset_summary = preview_dataset(values["data_folder"])
        values["max_steps"] = steps_for_epochs(dataset_summary["num_transitions"], values["batch_size"], epochs)

    return TrainConfig(**values)

def train_one(algorithm: str, epochs: Optional[int] = None, **overrides) -> Path:
    algorithm = algorithm.lower()
    config = make_config(algorithm, epochs=epochs, **overrides)
    print(f"Training {algorithm.upper()} with config:")
    print(json.dumps(asdict(config), indent=2))
    final_path = TRAINERS[algorithm](config)
    print(f"Saved final {algorithm.upper()} model to {final_path}")
    return final_path

if PREVIEW_DATASET and DATA_FOLDER.exists():
    dataset_summary = preview_dataset(DATA_FOLDER)
elif DATA_FOLDER.exists():
    print(f"Dataset folder found: {DATA_FOLDER}")
else:
    print(f"Set DATA_FOLDER to your dataset path. Current value does not exist: {DATA_FOLDER}")


## 8. Training Cell

Set algorithm_to_train to bc, awac, iql, or td3bc, then run this cell. Use the commented overrides as examples.


In [ ]:
algorithm_to_train = "bc"

final_model_path = train_one(
    algorithm_to_train,
    # epochs=10,
    # max_steps=10_000,
    # lr=1e-3,                 # BC only
    # awac_lambda=3.0,          # AWAC only
    # iql_tau=0.8, beta=3.0,    # IQL only
    # alpha=1.0,                # TD3+BC only
)

final_model_path


## 9. Optional: Train All Four Models

Run this cell only when you want a complete comparison. Each algorithm saves to its own folder under CHECKPOINT_ROOT.


In [ ]:
RUN_ALL = False

if RUN_ALL:
    final_paths = {}
    for algorithm in ["bc", "awac", "iql", "td3bc"]:
        final_paths[algorithm] = train_one(algorithm)
    final_paths
else:
    print("Set RUN_ALL = True to train BC, AWAC, IQL, and TD3+BC in sequence.")


## 10. Checkpoints and Download

Checkpoints are saved under CHECKPOINT_ROOT. In Colab, you can zip the folder and download it after training.


In [ ]:
def list_saved_models(root: str | Path = CHECKPOINT_ROOT) -> list[Path]:
    root = Path(root)
    if not root.exists():
        print(f"No checkpoint folder found yet: {root}")
        return []
    paths = []
    for path in sorted(root.rglob("*.pt")):
        if path.name.startswith("final_") or path.name.startswith("best_"):
            paths.append(path)
            print(path)
    return paths

def zip_checkpoints(root: str | Path = CHECKPOINT_ROOT, zip_path: str | Path = "/content/offline_controller_runs.zip") -> Path:
    root = Path(root)
    zip_path = Path(zip_path)
    if zip_path.exists():
        zip_path.unlink()
    shutil.make_archive(str(zip_path.with_suffix("")), "zip", root)
    print(f"Wrote {zip_path}")
    return zip_path

saved_models = list_saved_models()

# In Colab, uncomment these after training if you want a local download.
# zip_path = zip_checkpoints()
# from google.colab import files
# files.download(str(zip_path))
